In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("14-partitioning-shuffles")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]

# ── Section 1: Default partitions and current partition count ─────────────────
# spark.sql.shuffle.partitions controls post-shuffle partition count (default 200; overridden to 4)
# rdd.getNumPartitions() reports the CURRENT partition count of a DataFrame
print("\n=== Section 1: Default partitions ===")
print("Default shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("emp partitions:", emp.rdd.getNumPartitions())
print("sal partitions:", sal.rdd.getNumPartitions())
# UI: no job triggered yet; these are metadata-only calls

# ── Section 2: repartition (increase or redistribute) ────────────────────────
# repartition(N) → full shuffle (Exchange in the plan); all N partitions are roughly equal
print("\n=== Section 2: repartition(8) ===")
emp_repartitioned = emp.repartition(8)
print("After repartition(8):", emp_repartitioned.rdd.getNumPartitions())
emp_repartitioned.count()  # triggers the action; Exchange visible in Spark UI Stages
# UI: Stages view shows Exchange (shuffle write → shuffle read) before the count

# ── Section 3: repartition by column (hash partitioning on key) ──────────────
# repartition(N, col) hashes col and routes each row to hash(col) % N
# All rows with the same dept_id are guaranteed to land in the same partition
print("\n=== Section 3: repartition by dept_id ===")
emp_by_dept = emp.repartition(4, "dept_id")
print("Repartition by dept_id partitions:", emp_by_dept.rdd.getNumPartitions())
# Good pattern: pre-partition by the join/groupBy key to avoid downstream shuffle

# ── Section 4: coalesce (reduce partitions WITHOUT full shuffle) ──────────────
# coalesce(N) is a narrow transformation — no shuffle, just merges adjacent partitions
# Downside: partition sizes will be uneven (larger partitions absorb smaller ones)
print("\n=== Section 4: coalesce(2) ===")
emp_small = emp.coalesce(2)
print("After coalesce(2):", emp_small.rdd.getNumPartitions())
# UI: no Exchange in the plan — coalesce shows as a narrow stage boundary

# ── Section 5: repartition vs coalesce comparison ────────────────────────────
# repartition(N): full shuffle, balanced partitions, expensive
# coalesce(N):    narrow, may be imbalanced, cheap
# Rule: to REDUCE partitions after a filter → use coalesce
# Rule: to INCREASE or evenly redistribute → use repartition
print("\n=== Section 5: repartition vs coalesce ===")
print("repartition → full shuffle, balanced. coalesce → narrow, imbalanced but cheap.")
print("Use coalesce to reduce after filter; use repartition to increase or rebalance.")

# ── Section 6: shuffle.partitions tuning for small datasets ───────────────────
# Default shuffle.partitions=200 creates 200 tiny partitions on small data → overhead
# Set to a small number (4) matching parallelism for dev / small datasets
print("\n=== Section 6: shuffle.partitions tuning ===")
spark.conf.set("spark.sql.shuffle.partitions", "4")
emp_grouped = emp.groupBy("dept_id").agg(F.avg("salary").alias("avg_salary"))
print("Grouped partitions:", emp_grouped.rdd.getNumPartitions())  # should be 4 now
emp_grouped.show()
# UI: Stages shows Exchange with 4 output partitions (not 200)

# ── Section 7: Check partition sizes (data skew) ─────────────────────────────
# spark_partition_id() returns the partition index for each row
# Engineering (dept_id=1) has 14/41 rows = 34% → one partition gets much more data
print("\n=== Section 7: Partition sizes after repartition by dept_id ===")
emp_by_dept.withColumn("partition_id", F.spark_partition_id()) \
           .groupBy("partition_id").count() \
           .orderBy("partition_id").show()
# UI: uneven row counts across partitions = skew; the largest partition = stragglers in Stages view

# ── Section 8: Write partitioned by column (partition files on disk) ──────────
# partitionBy() creates subdirectories status=Active/ and status=Terminated/
# Enables partition pruning on read — Spark skips directories whose predicate doesn't match
print("\n=== Section 8: Write partitioned by status ===")
out = output_path("parquet/emp_by_status")
emp.write.mode("overwrite").partitionBy("status").parquet(out)
for fname in os.listdir(out):
    print(fname)
# Expected: status=Active/ and status=Terminated/ subdirectories (plus _SUCCESS)

# ── Section 9: Read a single partition (partition pruning) ────────────────────
# Spark reads only the matching subdirectory; the filter is pushed down to FileScan
print("\n=== Section 9: Read single partition (partition pruning) ===")
spark.read.parquet(out).filter(F.col("status") == "Terminated").show()
# UI: SQL tab → FileScan shows PartitionFilters = [status = Terminated]
# Only status=Terminated/ directory is read; other directories are skipped entirely

# ── Section 10: Detecting shuffle in explain plan ─────────────────────────────
# Exchange in the physical plan = a shuffle step; each Exchange means data crosses the network
print("\n=== Section 10: Shuffle in explain plan ===")
emp.join(sal, "emp_id", "inner").explain()
# Look for: Exchange hashpartitioning(emp_id, N) on both sides of the join
# UI: Stages view — the Exchange stage shows shuffle write bytes and shuffle read bytes

stop_and_wait(spark)